In [1]:
# on colab, mount google drive and reset this dir, else point to local dir
DATA_DIR = 'drive/MyDrive/ml-assignment-2'

In [ ]:
import pandas as pd
import numpy as np
import gc
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')
# --- 1. DATA PREP ---
print("Loading and scaling data...")
# BUG FIXED for splitting issue
df_train = pd.read_parquet(f'{DATA_DIR}/iceee_feature_train.parquet')
df_val   = pd.read_parquet(f'{DATA_DIR}/iceee_feature_val.parquet')

DROP_COLS = ['isFraud', 'TransactionID', 'TransactionDT']
X_train_raw = df_train.drop(DROP_COLS, axis=1, errors='ignore').fillna(-999)
X_val_raw   = df_val.drop(DROP_COLS, axis=1, errors='ignore').fillna(-999)
y_train = df_train['isFraud'].values
y_val   = df_val['isFraud'].values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_val = scaler.transform(X_val_raw).astype(np.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
del df_train, df_val, X_train_raw, X_val_raw; gc.collect()

# --- 2. THE AUTOENCODER REGIMES ---
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, 16))
        self.decoder = nn.Sequential(nn.Linear(16, 64), nn.ReLU(), nn.Linear(64, input_dim))
    def forward(self, x): return self.decoder(self.encoder(x))

def train_ae(train_data, epochs=10):
    model = Autoencoder(X_train.shape[1]).to(device)
    loader = DataLoader(TensorDataset(torch.from_numpy(train_data)), batch_size=2048, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(epochs):
        for batch in loader:
            x_b = batch[0].to(device)
            loss = nn.MSELoss()(model(x_b), x_b)
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        val_t = torch.from_numpy(X_val).to(device)
        recon = model(val_t)
        scores = torch.mean((val_t - recon)**2, dim=1).cpu().numpy()
    return scores, model  # return both to visualise latent space

# Regime A: Semi-supervised (Train on NORMAL only)
print(">>> Training AE: Semi-supervised Regime...")
ae_semi_scores, ae_semi_model = train_ae(X_train[y_train == 0])

# Regime B: Supervised (Train on ALL data)
print(">>> Training AE: Supervised Regime...")
ae_supp_scores = train_ae(X_train)

# --- 3. SUPERVISED MLP ---
print("\n>>> Training Supervised MLP...")
print("\n>>> Training Supervised MLP (Full Labeled Set)")
# Handling imbalance via weighted loss
pos_weight = torch.tensor([len(y_train[y_train==0]) / len(y_train[y_train==1])]).to(device)
mlp_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train.astype(np.float32))), batch_size=2048, shuffle=True)

class SupervisedMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1)  # CHANGED from earlier version: No Sigmoid because BCEWithLogitsLoss applies it internally
        )
    def forward(self, x): return self.net(x)

mlp = SupervisedMLP(X_train.shape[1]).to(device)
mlp_opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
# BCE with Pos Weight helps with the rare fraud class
mlp_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

for epoch in range(10):
    mlp.train()
    for xb, yb in mlp_loader:
        out = mlp(xb.to(device)).squeeze()
        loss = mlp_criterion(out.squeeze(), yb.to(device)) # using MLP criterion for class imbalance
        mlp_opt.zero_grad(); loss.backward(); mlp_opt.step()
    print(f"MLP Epoch {epoch+1}/10 complete")

mlp.eval()
with torch.no_grad():
    # CHANGED from v1: ADD back sigmoid here so that the final probabilities are intepretable
    mlp_probs = torch.sigmoid(mlp(torch.from_numpy(X_val).to(device))).cpu().numpy().flatten()

# --- 4. EXPORT ---
results_dict = {
    'actual': y_val,
    'ae_semi_supervised': ae_semi_scores, # Regime A semi supervised
    'ae_supervised': ae_supp_scores,      # Regime B supervised
    'mlp_prob': mlp_probs
}

# Save to CSV
df_final = pd.DataFrame(results_dict)
df_final.to_csv(f'{DATA_DIR}/dl_results.csv', index=False)

print("\nSuccess! Results saved to 'dl_results.csv' with all regimes.")

Loading and scaling data...
>>> Training AE: Semi-supervised Regime...
>>> Training AE: Supervised Regime...

>>> Training Supervised MLP...

>>> Training Supervised MLP (Full Labeled Set)
MLP Epoch 1/10 complete
MLP Epoch 2/10 complete
MLP Epoch 3/10 complete
MLP Epoch 4/10 complete
MLP Epoch 5/10 complete
MLP Epoch 6/10 complete
MLP Epoch 7/10 complete
MLP Epoch 8/10 complete
MLP Epoch 9/10 complete
MLP Epoch 10/10 complete

Success! Results saved to 'dl_results.csv' with all regimes.


In [ ]:
# latent space visualisation (semi-supervised model)
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

print(">>> Generating latent space visualisation...")
with torch.no_grad():
    val_t = torch.from_numpy(X_val).to(device)
    latent = ae_semi_model.encoder(val_t).cpu().numpy()  # (n_val, 16)

# subsample for t-SNE speed
sample_size = 5000
idx = np.random.choice(len(latent), sample_size, replace=False)
latent_2d = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(latent[idx])

plt.figure(figsize=(9, 7))
scatter = plt.scatter(latent_2d[:, 0], latent_2d[:, 1],
                      c=y_val[idx], cmap='RdYlGn_r', alpha=0.5, s=8)
plt.colorbar(scatter, label='0 = Legitimate, 1 = Fraud')
plt.title('t-SNE of Autoencoder Latent Space (semi-supervised)')
plt.xlabel('t-SNE dimension 1')
plt.ylabel('t-SNE dimension 2')
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/report/latent_space_tsne.png', dpi=150, bbox_inches='tight')
plt.show()